In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import geopandas as gpd
from rasterstats import zonal_stats
import rioxarray
import xvec
from rasterio.transform import Affine
import matplotlib.pyplot as plt
from rasterio.plot import show

Paths to metadata

In [1]:
meta_data = "C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_CASC_Project/dual-risk-repo/Climate_Extremes/climate_threshold_extreme_variables.csv"
meta_data

'C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_CASC_Project/dual-risk-repo/Climate_Extremes/climate_threshold_extreme_variables.csv'

Share socioeconomic pathway scenarios

In [2]:
scenarios = np.array(["ssp245","ssp370","ssp585"])
n_s = len(scenarios)

NameError: name 'np' is not defined

Future year intervals

In [ ]:
intervals = np.array(["2041_2070","2071_2100"])
n_y = len(intervals)

Temperature extreme variables

In [ ]:
meta_data_df = pd.read_csv(meta_data)
temp_variables = meta_data_df["Temperature"].to_numpy(dtype=str)
temp_variables = temp_variables[temp_variables != 'nan']

Precipitaton extreme variables

In [ ]:
precip_variables = meta_data_df["Precipitation"].to_numpy(dtype=str)

Climate threshold and extreme category

In [ ]:
threshold_categories = np.array(["pr","temp"])

County zone shapefile

In [ ]:
county_shapefile = "F:/Databases/Census/tl_2023_us_county/tl_2023_il_county_clipped_project.shp"
county_polygons = gpd.read_file(county_shapefile)

In [ ]:
category = "pr"
scenario = "ssp245"
interval = "2041_2070"
variable = "pr_annual"

In [ ]:
# make file path
category_path = f"F:/Wetland_Climate_Impacts/Climate_Data/USGS_Thresholds_Weighted/Grid/{category}/bma_weighted_differences/{scenario}/{interval}"
file_name = f"NCA5_BMA_Weighted_Ensemble_Difference_{variable}_{scenario}_{interval}.nc"
file_path = category_path + "/" + file_name

In [ ]:
# open raster
da = xr.open_dataset(file_path, engine="netcdf4") #"h5netcdf" "rasterio" 
da.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
da.rio.write_crs("EPSG:4326", inplace=True) 
da.rio.write_crs(4326,inplace=True).rio.set_spatial_dims(
    x_dim="lon",
    y_dim="lat",
    inplace=True
).rio.write_coordinate_system(inplace=True)
da.pr_annual.plot()

In [ ]:
# fix coordinate system
county_polygons.crs = da.rio.crs
da.rio.crs == county_polygons.crs

In [ ]:
county_polygons['geometry'] = county_polygons['geometry'].buffer(0)
county_polygons = county_polygons[~county_polygons['geometry'].is_empty]
#list(county_polygons.iloc[0].geometry.exterior.coords)

In [ ]:
da

In [ ]:
county_polygons.plot()

In [ ]:
da.pr_annual.values

In [ ]:
da_test = xr.DataArray(
    data=da.pr_annual.values,
    dims=["x", "y"],
    coords=dict(
        lon=("x", da.lon),
        lat=("y", da.lat),
    ),
    name="pr_annual",
)

In [ ]:

zone_stats = da_dropped_spatial_ref.xvec.zonal_stats(county_polygons, 
                                 x_coords="lat",
                                 y_coords="lon",
                                 stats="mean")

In [ ]:
# Load raster
#for category in threshold_categories:
#    for scenario in scenarios: 
#        if category == "pr":
#            for variable in precip_variables:
#                for interval in intervals:            